# Cell 6 — Memory-Safe Pipeline
Processes regions in small batches to avoid RAM exhaustion.
Uses float32 instead of float64 to halve memory usage.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import os, gc, time
from tqdm.notebook import tqdm

# ── CONFIG ────────────────────────────────────────────────────────────
SIMULATION_DIR   = './GCM_models/CNRM-ESM2-1/RegridedMonthly'
TARGET_DIR       = './GCM_models/CNRM-ESM2-1/regional_averages_output'
MASKS_PATH       = './masks/subregion_masks.nc'
INDICATOR        = 'tas'
SCENARIOS_I_WANT = ['historical','ssp126','ssp245','ssp370','ssp460','ssp585']

# ── KEY SETTING: how many regions to process at once ─────────────────
# Lower = less RAM used. Start with 50, raise if you have headroom.
# At 50 regions/batch: ~1-2GB RAM per batch (safe on 140GB server)
BATCH_SIZE = 50

print(f"Batch size: {BATCH_SIZE} regions per batch")
print(f"Total regions in mask: will be shown after mask loads")


In [ ]:
# ── LOAD SIMULATION FILE ─────────────────────────────────────────────
# Load the NetCDF ONCE and keep it in memory
# Only load the indicator variable, not the whole dataset

all_files = [
    f for f in os.listdir(SIMULATION_DIR)
    if f.endswith('.nc')
    and len(f.split('_')) > 3
    and f.split('_')[3] in SCENARIOS_I_WANT
]

print(f"Files found: {len(all_files)}")
for f in all_files:
    print(f"  {f}")


In [ ]:
# ── LOAD MASK VARIABLE NAMES ONLY (no data yet) ──────────────────────
# We open the mask dataset but DON'T load all variables into memory
# We will load them in small batches

masks_ds = xr.open_dataset(MASKS_PATH)
all_region_names = list(masks_ds.data_vars)

print(f"Total regions in mask : {len(all_region_names)}")
print(f"Sample names          : {all_region_names[:5]}")
print(f"Batch size            : {BATCH_SIZE}")
print(f"Total batches per file: {len(all_region_names) // BATCH_SIZE + 1}")

# Load lat/lon coords for cos(lat) weighting
lat_vals = masks_ds.lat.values
cos_lat_1d = np.cos(np.deg2rad(lat_vals)).astype(np.float32)
print(f"\ncos(lat) weights ready. Shape: {cos_lat_1d.shape}")


In [ ]:
# ── MEMORY-SAFE PIPELINE FUNCTION ────────────────────────────────────

def process_file_in_batches(
    simulation_directory,
    simulation_file,
    target_directory,
    masks_path,
    all_region_names,
    cos_lat_1d,
    indicator,
    batch_size=50,
    need_global_mean=False,
):
    simulation_path = f"{simulation_directory}/{simulation_file}"

    MODEL     = simulation_file.split('_')[2]
    scenario  = simulation_file.split('_')[3]
    ensemble  = simulation_file.split('_')[4]
    SCENARIO  = f"{scenario}-{ensemble}"
    INDICATOR = simulation_file.split('_')[0]

    print(f"\nProcessing: {MODEL} | {SCENARIO} | {INDICATOR}")

    # ── Load simulation — only the indicator variable, cast to float32
    print("  Loading simulation NetCDF...")
    sim_ds = xr.open_dataset(simulation_path)

    # Fix lon: 0-360 → -180 to 180
    sim_ds['lon'] = xr.where(
        sim_ds['lon'] > 180,
        sim_ds['lon'] - 360,
        sim_ds['lon']
    )
    sim_ds = sim_ds.sortby('lon')

    # Load ONLY the indicator into memory as float32 (halves RAM vs float64)
    sim_data = sim_ds[indicator].load().astype(np.float32)
    time_vals = sim_ds['time'].values
    sim_ds.close()

    print(f"  Simulation loaded: shape={sim_data.shape}, dtype={sim_data.dtype}")

    # ── Precompute cos(lat) broadcast array (shape: lat, lon)
    cos_lat_2d = np.broadcast_to(
        cos_lat_1d[:, np.newaxis],
        (len(cos_lat_1d), sim_data.shape[2])
    ).astype(np.float32)

    # ── Handle global mean separately (no batch needed) ──────────────
    if need_global_mean:
        global_avg = (
            (sim_data.values * cos_lat_2d[np.newaxis, :, :]).sum(axis=(1, 2))
            / cos_lat_2d.sum()
        )
        df_global = pd.DataFrame({'time': time_vals, 'tas': global_avg})
        gm_dir = f"{target_directory}/cmip-6_global_mean/{indicator}"
        os.makedirs(gm_dir, exist_ok=True)
        df_global.to_csv(
            f"{gm_dir}/globalmean_{INDICATOR.lower()}_{MODEL.lower()}_{SCENARIO.lower()}.csv",
            index=False
        )
        print(f"  Global mean saved ✅")
        del global_avg, df_global

    # ── Process regions in batches ────────────────────────────────────
    n_batches = len(all_region_names) // batch_size + 1
    sim_np = sim_data.values   # shape: (time, lat, lon) — numpy, faster

    for batch_idx in tqdm(range(n_batches), desc=f"  Batches ({MODEL})"):

        batch_regions = all_region_names[
            batch_idx * batch_size : (batch_idx + 1) * batch_size
        ]
        if not batch_regions:
            break

        # Load only this batch of mask variables
        batch_masks_ds = xr.open_dataset(
            masks_path,
            drop_variables=[r for r in all_region_names if r not in batch_regions]
        ).load()

        for region in batch_regions:
            region_clean = region.replace('m_', '')

            # Get mask as float32 numpy array (lat, lon)
            mask_np = batch_masks_ds[region].values.astype(np.float32)

            # Apply cos(lat) weighting to mask
            weighted_mask = mask_np * cos_lat_2d   # (lat, lon)

            # Replace 0 with nan so they don't count in sum
            weighted_mask = np.where(weighted_mask == 0, np.nan, weighted_mask)

            # Area-weighted average per timestep:
            # sum(mask * data, axis=(lat,lon)) / sum(mask, axis=(lat,lon))
            mask_sum = np.nansum(weighted_mask)

            if mask_sum == 0:
                # Region has no valid grid cells at 2.5deg resolution — skip
                continue

            regional_ts = (
                np.nansum(sim_np * weighted_mask[np.newaxis, :, :], axis=(1, 2))
                / mask_sum
            )

            # Write CSV
            df = pd.DataFrame({
                'time':        time_vals,
                region_clean:  regional_ts,
            })
            out_dir = (
                f"{target_directory}/isimip_regional_data"
                f"/{region_clean}/latWeight"
            )
            filename = (
                f"{MODEL}_{SCENARIO}_{INDICATOR}_{region_clean}_latweight.csv"
                .lower()
            )
            os.makedirs(out_dir, exist_ok=True)
            df.to_csv(f"{out_dir}/{filename}", index=False)

            del mask_np, weighted_mask, regional_ts, df

        # Free batch mask memory before next batch
        batch_masks_ds.close()
        del batch_masks_ds
        gc.collect()

    # ── Cleanup simulation ────────────────────────────────────────────
    del sim_data, sim_np, cos_lat_2d
    gc.collect()
    print(f"  Done ✅  Regions written: {len(all_region_names)}")


print("Function defined ✅")


In [ ]:
# ── RUN THE PIPELINE ─────────────────────────────────────────────────

need_global_mean = (INDICATOR == 'tas')

print(f"Starting pipeline")
print(f"Files         : {len(all_files)}")
print(f"Regions       : {len(all_region_names)}")
print(f"Batch size    : {BATCH_SIZE} regions")
print(f"Global mean   : {need_global_mean}")
print("=" * 55)

for file in all_files:
    process_file_in_batches(
        simulation_directory = SIMULATION_DIR,
        simulation_file      = file,
        target_directory     = TARGET_DIR,
        masks_path           = MASKS_PATH,
        all_region_names     = all_region_names,
        cos_lat_1d           = cos_lat_1d,
        indicator            = INDICATOR,
        batch_size           = BATCH_SIZE,
        need_global_mean     = need_global_mean,
    )

print("=" * 55)
print("PIPELINE COMPLETE ✅")


In [ ]:
# ── VERIFY OUTPUTS ───────────────────────────────────────────────────
import os, pandas as pd

# Check global mean
gm_dir = f"{TARGET_DIR}/cmip-6_global_mean/{INDICATOR}"
if os.path.exists(gm_dir):
    gm_files = os.listdir(gm_dir)
    print(f"Global mean CSVs: {len(gm_files)}")
    for f in gm_files:
        df = pd.read_csv(f"{gm_dir}/{f}")
        print(f"  {f}")
        print(f"  rows={len(df)}, "
              f"years={df['time'].iloc[0]} → {df['time'].iloc[-1]}")

# Check regional data
reg_dir = f"{TARGET_DIR}/isimip_regional_data"
if os.path.exists(reg_dir):
    regions_written = os.listdir(reg_dir)
    print(f"\nRegions with output CSVs : {len(regions_written)}")
    print(f"Sample regions           : {regions_written[:5]}")

    # Preview one CSV
    if regions_written:
        r = regions_written[0]
        lw = f"{reg_dir}/{r}/latWeight"
        if os.path.exists(lw):
            f = os.listdir(lw)[0]
            df = pd.read_csv(f"{lw}/{f}")
            print(f"\nSample file: {f}")
            print(df.head(3).to_string(index=False))
